# 🖊️ Point-and-Read: Handwritten Text Recognition
### CRNN + BiLSTM + CTC | IAM Handwriting Database | UTS 42028 Deep Learning

---

> **Goal:** Train a Convolutional Recurrent Neural Network (CRNN) with CTC loss to recognise 
> handwritten English text from line images. The trained model powers a local accessibility 
> app that reads handwritten text aloud for visually impaired users.

| | |
|---|---|
| **Architecture** | ResNet-34 → BiLSTM (×2) → CTC Decoder |
| **Dataset** | IAM Handwriting Database — 9,045 train / 1,136 val / 1,163 test lines |
| **Framework** | PyTorch |
| **Hardware** | NVIDIA GPU (CUDA) |
| **Target Metric** | CER < 10%, WER < 25% on IAM test set |


## 📋 Table of Contents

1. [Environment Setup](#1-environment-setup)
2. [Data Pipeline](#2-data-pipeline)
   - 2.1 Extract IAM Data
   - 2.2 Verify Folder Structure
   - 2.3 Parse IAM → CSVs + Vocabulary
3. [Preprocessing Verification](#3-preprocessing-verification)
4. [Model Architecture](#4-model-architecture)
5. [Training](#5-training)
6. [Results & Evaluation](#6-results--evaluation)
7. [Inference Demo](#7-inference-demo)


---
## 1. Environment Setup
Clone the repository, install dependencies, and verify GPU availability.


In [ ]:
import os, sys

!git clone https://github.com/AmanSinghNp/point-and-read.git /kaggle/working/point-and-read -q
!git -C /kaggle/working/point-and-read pull

os.chdir("/kaggle/working/point-and-read")
sys.path.insert(0, "/kaggle/working/point-and-read")

!pip install editdistance -q

import torch
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
print(f"Device          : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")


---
## 2. Data Pipeline

The IAM Handwriting Database contains 13,353 pre-cropped line images from 657 writers.
Line images follow a three-level nested folder structure:

lines/
└── a01/ ← writer prefix
└── a01-000u/ ← form ID
├── a01-000u-00.png
└── a01-000u-01.png


Splits are **form-based** (not random) to prevent the same writer's style appearing in 
both train and test sets — ensuring honest CER/WER numbers.

### 2.1 Extract IAM Data


In [ ]:
import os

DATASET_PATH = "/kaggle/input/datasets/amansinghnp/kaggleinputiam-handwriting-lines"

os.makedirs("/kaggle/working/point-and-read/data/iam", exist_ok=True)

!tar -xzf {DATASET_PATH}/ascii.tgz -C /kaggle/working/point-and-read/data/iam/
!tar -xzf {DATASET_PATH}/lines.tgz -C /kaggle/working/point-and-read/data/iam/

print("✅ Extracted:", os.listdir("/kaggle/working/point-and-read/data/iam"))


### 2.2 Verify Folder Structure
Confirm the three-level image path resolves correctly before parsing.


In [ ]:
import os

# Find where the actual .png files are
for root, dirs, files in os.walk("/kaggle/working/point-and-read/data/iam"):
    pngs = [f for f in files if f.endswith('.png')]
    if pngs:
        print(root)
        print("  Sample:", pngs[:3])
        break


In [ ]:
import shutil

src = "/kaggle/working/point-and-read/data/iam"
dst = "/kaggle/working/point-and-read/data/iam/lines"

os.makedirs(dst, exist_ok=True)

# Move all writer folders (a01, a02, etc.) into lines/
for item in os.listdir(src):
    item_path = os.path.join(src, item)
    # Only move directories that look like writer IDs (e.g. a01, b02)
    if os.path.isdir(item_path) and item != "lines":
        shutil.move(item_path, os.path.join(dst, item))
        print(f"Moved: {item} → lines/{item}")

print("\nlines/ now contains:", os.listdir(dst)[:5])


In [ ]:
!python scripts/verify_iam_structure.py \
  --lines_txt data/iam/lines.txt \
  --iam_root  data/iam \
  --n 10

### 2.3 Parse IAM → CSVs + Vocabulary

`parse_iam.py` produces:
- `data/processed/train.csv` — 9,045 lines (80%)
- `data/processed/val.csv` — 1,136 lines (10%)  
- `data/processed/test.csv` — 1,163 lines (10%)
- `data/processed/vocab.json` — 79 characters, built from train only (no data leakage)


In [ ]:
!python parse_iam.py \
  --lines_txt data/iam/lines.txt \
  --iam_root  data/iam \
  --out_dir   data/processed

import pandas as pd

train_df = pd.read_csv("data/processed/train.csv", header=None, names=["image_path", "transcription"])
val_df   = pd.read_csv("data/processed/val.csv",   header=None, names=["image_path", "transcription"])
test_df  = pd.read_csv("data/processed/test.csv",  header=None, names=["image_path", "transcription"])

print(f"\n{'Split':<10} {'Samples':>10} {'Avg chars':>12}")
print("-" * 34)
print(f"{'Train':<10} {len(train_df):>10,} {train_df['transcription'].str.len().mean():>12.1f}")
print(f"{'Val':<10} {len(val_df):>10,}   {val_df['transcription'].str.len().mean():>12.1f}")
print(f"{'Test':<10} {len(test_df):>10,}   {test_df['transcription'].str.len().mean():>12.1f}")

# Preview a few samples
print("\nSample rows:")
print(train_df.head(3).to_string())



---
## 3. Preprocessing Verification

Each image passes through: **Grayscale → Denoise → Binarize (Otsu) → Resize to 64px height → Normalize**

Deskew is disabled for IAM (lines are pre-cropped and already straight).
The output below shows 5 sample preprocessed images for visual inspection.


In [ ]:
!python scripts/verify_preprocessing.py \
  --csv     data/processed/train.csv \
  --out_dir data/processed/preview \
  --n 5

from IPython.display import Image, display
import glob

previews = sorted(glob.glob("data/processed/preview/*no_deskew*"))[:5]
for p in previews:
    print(os.path.basename(p))
    display(Image(p, width=700))


---
## 4. Model Architecture

Input (B, 1, 64, W)
↓
ResNet-34 CNN (ImageNet init, 1-channel input)
[Pools height to 1, preserves width sequence]
↓
BiLSTM × 2 (hidden=256, dropout=0.3)
[Reads feature sequence bidirectionally]
↓
Linear → log_softmax (79 + 1 classes)
[+1 for CTC blank token at index 0]
↓
CTC Loss / CTC Greedy Decoder
↓
Output: recognised text string


| Component | Details |
|---|---|
| CNN Backbone | ResNet-34, pretrained on ImageNet |
| Input channels | 1 (grayscale) |
| Fixed height | 64px |
| Width stride | ×32 reduction (sequence length = W/32) |
| BiLSTM layers | 2 × bidirectional, hidden size 256 |
| Output classes | 80 (79 chars + CTC blank) |
| Total parameters | ~24.4M |



In [ ]:
from recognition.model import CRNN
from recognition.vocab import Vocabulary

vocab = Vocabulary.load("data/processed/vocab.json")
model = CRNN(num_classes=vocab.num_classes)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters     : {total:,}")
print(f"Trainable parameters : {trainable:,}")
print(f"Vocabulary size      : {vocab.num_classes} classes ({vocab.num_classes-1} chars + 1 blank)")


---
## 5. Training

**Configuration:**

| Hyperparameter | Value |
|---|---|
| Optimiser | Adam |
| Learning rate | 1e-3 |
| LR schedule | ReduceLROnPlateau (factor=0.5, patience=5) |
| Batch size | 32 |
| Gradient clipping | max_norm = 5.0 |
| Early stopping | patience = 15 epochs |
| Max epochs | 50 |
| CTC blank index | 0 |

Training logs are saved to `weights/training_log.csv` after each epoch.
Best checkpoint (by validation CER) is saved to `weights/best.pth`.


In [ ]:
# Add this cell BEFORE the training cell and run it
import recognition.train as _train_module
import torch, inspect

# Monkey-patch the scheduler line
original_train = _train_module.train.__code__

# Easier: just edit the file directly in Kaggle
with open("recognition/train.py", "r") as f:
    src = f.read()

src = src.replace(
    'optimizer, mode="min", factor=0.5, patience=5, verbose=True',
    'optimizer, mode="min", factor=0.5, patience=5'
)

with open("recognition/train.py", "w") as f:
    f.write(src)

print("✅ Fixed. Reloading module...")

import importlib
import recognition.train
importlib.reload(recognition.train)
from recognition.train import train
print("✅ Module reloaded.")


In [ ]:
from recognition.train import train

config = {
    "train_csv":    "data/processed/train.csv",
    "val_csv":      "data/processed/val.csv",
    "vocab_path":   "data/processed/vocab.json",
    "save_dir":     "weights",
    "log_csv":      "weights/training_log.csv",
    "epochs":        50,
    "batch_size":    32,
    "lr":            1e-3,
    "weight_decay":  1e-4,
    "grad_clip":     5.0,
    "patience":      15,
    "apply_deskew":  False,
}

os.makedirs("weights", exist_ok=True)
train(config)


In [ ]:
# Quick sanity check on sequence lengths
import pandas as pd
df = pd.read_csv("data/processed/train.csv", header=None, names=["image_path","transcription"])
print("Max transcription length:", df["transcription"].str.len().max())
print("Avg transcription length:", df["transcription"].str.len().mean())


---
## 6. Results & Evaluation

Plot training curves and evaluate the best checkpoint on the held-out test set.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

log = pd.read_csv("weights/training_log.csv")

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("Training Curves — CRNN + BiLSTM + CTC", fontsize=14, fontweight="bold")

axes[0].plot(log["epoch"], log["train_loss"], label="Train Loss", color="#4C72B0")
axes[0].plot(log["epoch"], log["val_loss"],   label="Val Loss",   color="#DD8452")
axes[0].set_title("Loss"); axes[0].legend(); axes[0].set_xlabel("Epoch")

axes[1].plot(log["epoch"], log["val_cer"], color="#55A868")
axes[1].set_title("Validation CER"); axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("CER")
axes[1].axhline(0.10, color="red", linestyle="--", label="Target (10%)")
axes[1].legend()

axes[2].plot(log["epoch"], log["lr"], color="#C44E52")
axes[2].set_title("Learning Rate"); axes[2].set_xlabel("Epoch")

plt.tight_layout()
plt.savefig("weights/training_curves.png", dpi=150)
plt.show()

best_epoch = log.loc[log["val_cer"].idxmin()]
print(f"\nBest Epoch   : {int(best_epoch['epoch'])}")
print(f"Best Val CER : {best_epoch['val_cer']:.4f}")
print(f"Best Val WER : {best_epoch['val_wer']:.4f}")


---
## 7. Inference Demo

Run the trained model on sample test images to visually verify predictions.


In [ ]:
from recognition.inference import load_model, recognise_line, Vocabulary, Preprocessor
from IPython.display import Image, display
import pandas as pd
import torch

vocab = Vocabulary.load("data/processed/vocab.json")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = load_model("weights/best.pth", vocab, device)
preprocessor = Preprocessor()

test_df = pd.read_csv("data/processed/test.csv", header=None, names=["image_path", "transcription"]).head(5)

print(f"{'Ground Truth':<45} {'Prediction':<45}")
print("-" * 90)
for _, row in test_df.iterrows():
    pred = recognise_line(row["image_path"], model, vocab, preprocessor, device)
    gt   = row["transcription"]
    match = "✅" if pred.strip() == gt.strip() else "❌"
    print(f"{match} GT  : {gt}")
    print(f"   PRED: {pred}\n")
    display(Image(row["image_path"], width=600))


In [ ]:
!pip install pyctcdecode -q
from pyctcdecode import build_ctcdecoder
from recognition.inference import Vocabulary, Preprocessor, CRNN
import torch
import numpy as np
import pandas as pd

# Setup
vocab = Vocabulary.load("data/processed/vocab.json")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
preprocessor = Preprocessor()

model = CRNN(num_classes=vocab.num_classes).to(device)
ckpt = torch.load("weights/best.pth", map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

# Build beam decoder — blank at index 0
labels = [''] + list(vocab.chars)
decoder = build_ctcdecoder(labels)

# Eval on test set
test_df = pd.read_csv("data/processed/test.csv", header=None, names=["image_path", "transcription"]).head(20)

print(f"{'GT':<45} {'Greedy':<35} {'Beam':<35}")
print("-" * 115)

for _, row in test_df.iterrows():
    tensor = preprocessor.process(row["image_path"]).unsqueeze(0).to(device)  # (1,1,H,W)

    with torch.no_grad():
        log_probs = model(tensor)  # (T, 1, C)

    # Greedy decode
    greedy_indices = log_probs.squeeze(1).argmax(-1).cpu().tolist()
    greedy_chars, prev = [], None
    for idx in greedy_indices:
        if idx != prev and idx != 0:
            greedy_chars.append(vocab.chars[idx - 1] if idx > 0 else '')
        prev = idx
    greedy_pred = ''.join(greedy_chars)

    # Beam decode
    logits_np = log_probs.squeeze(1).cpu().numpy()  # (T, C)
    beam_pred = decoder.decode(logits_np, beam_width=25)

    gt = row["transcription"]
    print(f"GT  : {gt[:43]}")
    print(f"PRED: {greedy_pred[:43]:<43} BEAM: {beam_pred[:43]}")
    print()


#Trying new model



In [4]:
test_df = pd.read_csv("/kaggle/working/point-and-read/data/processed/test.csv", 
                      header=None,
                      names=["image_path", "transcription"]).head(10)


In [5]:
for _, row in test_df.iterrows():
    img_path = "/kaggle/working/point-and-read/" + row["image_path"]
    img = PILImage.open(img_path).convert("RGB")
    pixel_values = processor(img, return_tensors="pt").pixel_values
    with torch.no_grad():
        generated_ids = trocr.generate(pixel_values)
    pred = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    print(f"GT  : {row['transcription'][:48]}")
    print(f"PRED: {pred[:48]}\n")


GT  : than three thousand three fifty at the most . No
PRED: than three thousand three fifty at the most . No

GT  : could get quite a nice little semi-detached hous
PRED: could get quite a nice little semi-debashed hous

GT  : Grimstead for three thousand , that 's where I l
PRED: Grimstead for three thousand , that's were I liv

GT  : before the green belt , lovely and modern , you 
PRED: before the green belt , lovely and modern , you 

GT  : I forgot to tell you , we don't usually lend any
PRED: forgot to tell you , we don't usually lend any

GT  : The housewife would find life far less tiring
PRED: The housewife would find his for her living

GT  : if she made a list , followed a routine of work
PRED: ijshe made a list , followed a routine of work

GT  : rather than getting from one thing to the next .
PRED: rather than getting from one thing to the next .

GT  : The business man would find that he reached
PRED: The business man would find that he reached

GT  : the end of the day

In [1]:
# Cell 1 — run this alone, then restart kernel
!pip install "numpy>=2.0" -q
!pip install transformers -q


In [3]:
!pip install transformers -q

from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from PIL import Image as PILImage
import pandas as pd
import torch

# Load model
processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-handwritten")
trocr = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-handwritten")
trocr.eval()

# Load test data
test_df = pd.read_csv("data/processed/test.csv", header=None,
                      names=["image_path", "transcription"]).head(10)

print(f"{'GT':<50} {'TrOCR':<50}")
print("-" * 100)
for _, row in test_df.iterrows():
    img = PILImage.open(row["image_path"]).convert("RGB")
    pixel_values = processor(img, return_tensors="pt").pixel_values
    with torch.no_grad():
        generated_ids = trocr.generate(pixel_values)
    pred = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    print(f"GT  : {row['transcription'][:48]}")
    print(f"PRED: {pred[:48]}\n")  # fixed escaped \n → actual newline


Loading weights:   0%|          | 0/478 [00:00<?, ?it/s]

VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-base-handwritten
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.bias   | MISSING | 
encoder.pooler.dense.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


FileNotFoundError: [Errno 2] No such file or directory: 'data/processed/test.csv'